# Logistic regression & cross-entropy

Logistic regression predicts a **probability** of class 1. It's exactly
[linear regression](linear-regression.ipynb)'s `w0 + w1·x` — squashed through a
**sigmoid** into `(0, 1)` — and trained by minimizing **cross-entropy**, not
squared error. This notebook builds up *what* is being optimized before calling
any `.fit()`.

In [ ]:
// All deps up front (adding a :dep later would make evcxr drop x/y).
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
:dep smartcore = { version = "0.3" }

// One feature; classes overlap near x=0 (label noise) so the fit is well-behaved.
let n = 80usize;
let x: Vec<f64> = (0..n).map(|i| (i as f64 / n as f64) * 6.0 - 3.0).collect();
let y: Vec<f64> = (0..n).map(|i| { let noise = (((i * 37) % 11) as f64 - 5.0) * 0.35; if x[i] + noise > 0.0 { 1.0 } else { 0.0 } }).collect();

fn sigmoid(z: f64) -> f64 { 1.0 / (1.0 + (-z).exp()) }
println!("{} samples; predict P(class 1) from one feature x", n);

## The sigmoid

The sigmoid maps any real number to a probability in `(0, 1)` — an S-curve. This
is the only thing separating logistic regression from linear regression:

In [ ]:
{
    use plotters::prelude::*;
    evcxr_figure((460, 240), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root).caption("sigmoid(z) = 1 / (1 + e^-z)", ("sans-serif", 15)).margin(8).x_label_area_size(30).y_label_area_size(40).build_cartesian_2d(-8f64..8f64, 0f64..1f64)?;
        c.configure_mesh().draw()?;
        c.draw_series(LineSeries::new((0..=160).map(|i| { let z = -8.0 + i as f64 * 0.1; (z, sigmoid(z)) }), &BLUE))?;
        Ok(())
    })
}

## Why not just reuse MSE?

Squared error on a sigmoid output gives a **non-convex** loss surface (flat
regions with vanishing gradients when the sigmoid saturates), which breaks the
convergence guarantees the [gradient-descent chapter](../05b-optimization/gradient-descent-variants.ipynb)
relied on. **Cross-entropy** is the fix — it's convex for logistic regression.

The per-example cross-entropy loss is `−[y·log(p) + (1−y)·log(1−p)]`. Let's see
what it actually costs for confident-correct, confident-wrong, and unsure
predictions:

In [ ]:
{
    let ce = |y: f64, p: f64| -(y * p.ln() + (1.0 - y) * (1.0 - p).ln());
    println!("true label y=1, predicted p=0.99 (confident, correct): loss = {:.3}", ce(1.0, 0.99));
    println!("true label y=1, predicted p=0.50 (unsure):             loss = {:.3}", ce(1.0, 0.50));
    println!("true label y=1, predicted p=0.01 (confident, WRONG):   loss = {:.3}", ce(1.0, 0.01));
    println!("-> being confidently wrong is punished far harder than being unsure.");
}

## Convexity, made visible

Sweep the coefficient `w1` and plot the total cross-entropy loss over the
dataset. It's a smooth, single-minimum **bowl** — convex, so gradient descent
can't get stuck:

In [ ]:
{
    use plotters::prelude::*;
    let total_ce = |w1: f64| -> f64 {
        (0..n).map(|i| { let p = sigmoid(w1 * x[i]).clamp(1e-6, 1.0 - 1e-6); -(y[i] * p.ln() + (1.0 - y[i]) * (1.0 - p).ln()) }).sum::<f64>() / n as f64
    };
    let ws: Vec<f64> = (0..=80).map(|i| -2.0 + i as f64 * 0.1).collect();
    let losses: Vec<f64> = ws.iter().map(|&w| total_ce(w)).collect();
    let (ymin, ymax) = (losses.iter().cloned().fold(f64::MAX, f64::min), losses.iter().cloned().fold(f64::MIN, f64::max));
    evcxr_figure((460, 260), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root).caption("Cross-entropy loss vs w1 (convex)", ("sans-serif", 15)).margin(8).x_label_area_size(30).y_label_area_size(45).build_cartesian_2d(-2f64..6f64, (ymin * 0.95)..(ymax * 1.05))?;
        c.configure_mesh().x_desc("w1").y_desc("loss").draw()?;
        c.draw_series(LineSeries::new(ws.iter().zip(&losses).map(|(&w, &l)| (w, l)), &BLUE))?;
        Ok(())
    })
}

## Fit it by hand with gradient descent

Cross-entropy has a beautifully simple gradient: for `p = sigmoid(w0 + w1·x)`, the
per-example gradient is just `(p − y)` (times `x` for `w1`). That's the same
[gradient descent](../05b-optimization/gradient-descent-variants.ipynb) loop as
linear regression, with cross-entropy's gradient swapped in:

In [ ]:
let hand_coef: (f64, f64) = {
    let (mut w0, mut w1) = (0.0, 0.0);
    let (lr, epochs) = (0.5, 4000);
    for _ in 0..epochs {
        let (mut g0, mut g1) = (0.0, 0.0);
        for i in 0..n { let e = sigmoid(w0 + w1 * x[i]) - y[i]; g0 += e; g1 += e * x[i]; }
        w0 -= lr * g0 / n as f64; w1 -= lr * g1 / n as f64;
    }
    let acc = (0..n).filter(|&i| ((sigmoid(w0 + w1 * x[i]) > 0.5) as i32 as f64) == y[i]).count() as f64 / n as f64;
    println!("hand-rolled: w0 = {:.3}, w1 = {:.3}, accuracy = {:.3}", w0, w1, acc);
    (w0, w1)
};

## Fit it with the library, and confirm they agree

`smartcore`'s `LogisticRegression` does the same optimization (more carefully).
The coefficients should land close to the hand-rolled ones — the
library-vs-from-scratch check:

In [ ]:
{
    use smartcore::linalg::basic::matrix::DenseMatrix;
    use smartcore::linalg::basic::arrays::Array;
    use smartcore::linear::logistic_regression::LogisticRegression;
    let xm = DenseMatrix::new(n, 1, x.clone(), false);
    let yc: Vec<u32> = y.iter().map(|&v| v as u32).collect();
    let lr = LogisticRegression::fit(&xm, &yc, Default::default()).unwrap();
    println!("library:     w0 = {:.3}, w1 = {:.3}", lr.intercept().get((0, 0)), lr.coefficients().get((0, 0)));
    println!("hand-rolled: w0 = {:.3}, w1 = {:.3}", hand_coef.0, hand_coef.1);
    println!("-> same story: both learn a positive slope on x (higher x -> class 1).");
}

## The decision threshold is a *separate* choice

Cross-entropy is minimized during **training** to produce calibrated
probabilities. Turning a probability into a 0/1 label needs a **threshold** —
0.5 is the default, but it's a *post-hoc* decision you tune for your costs (is a
false positive worse than a false negative?). That trade-off is exactly what the
[evaluation chapter's ROC and precision-recall curves](../01d-evaluation/metrics-deep-dive.ipynb)
are for.

Next: [regularized logistic regression](regularized-logistic-regression.ipynb) —
penalizing this cross-entropy loss to fight overfitting.